<a href="https://colab.research.google.com/github/karna-charan/LLM/blob/main/RAG2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pypdf sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.8 MB/s eta 0:00:00


In [ ]:
from pypdf import PdfReader
from google.colab import files
import os

# Upload the PDF file
print("Please upload your PDF document.")
uploaded = files.upload()

# Assuming only one file is uploaded, get its name
if uploaded:
    pdf_filename = list(uploaded.keys())[0]
    print(f"Successfully uploaded: {pdf_filename}")
else:
    print("No file was uploaded. Please try again.")
    # If no file is uploaded, the script cannot proceed.
    # You might want to raise an error or exit here.
    # For now, we'll assign a placeholder, which will likely lead to an error.
    pdf_filename = "document.pdf" # This will likely cause a FileNotFoundError if no file was uploaded.

# Check if the file exists after upload (for robustness)
if not os.path.exists(pdf_filename):
    print(f"Error: File '{pdf_filename}' not found after upload. Please ensure you uploaded the correct file.")
else:
    reader = PdfReader(pdf_filename)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    print(text[:500])   # preview first 500 characters

Please upload your PDF document.


Saving rag_sample_document (1).pdf to rag_sample_document (1).pdf
Successfully uploaded: rag_sample_document (1).pdf
Machine Learning Overview 
 
Machine Learning (ML) is a field of Artificial Intelligence that focuses 
on building systems that can learn from data and improve their 
performance over time without being explicitly programmed. 
 
Types of Machine Learning 
 
1.  Supervised Learning Supervised learning uses labeled data to train 
    models. The model learns the relationship between input features and 
    target outputs. Examples include: 
 
-   Linear Regression 
-   Logistic Regression 
-   Dec


In [ ]:
chunk_size = 500
chunks = []

for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i+chunk_size])

print(len(chunks))

5


In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(chunks)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import faiss
import numpy as np

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(doc_embeddings))

In [ ]:
query = input("Ask a question about the PDF: ")

query_embedding = embed_model.encode([query])

Ask a question about the PDF: What is deep learning?


In [ ]:
k = 3

distances, indices = index.search(query_embedding, k)

retrieved_docs = [chunks[i] for i in indices[0]]

context = " ".join(retrieved_docs)

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")

prompt = f"""
Context: {context}

Question: {query}

Answer:
"""

result = generator(prompt, max_length=200)

print(result[0]["generated_text"])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context: of machine learning that uses neural networks 
with many layers (deep neural networks). Applications include: - Image 
recognition - Speech recognition - Natural language processing 
 
Natural Language Processing 
 
Natural Language Processing (NLP) allows computers to understand, 
interpret, and generate human language. Common NLP tasks include: - Text 
classification - Sentiment analysis - Machine translation - Question 
answering 
 
Retrieval-Augmented Generation (RAG) 
 
RAG is an AI archite Machine Learning Overview 
 
Machine Learning (ML) is a field of Artificial Intelligence that focuses 
on building systems that can learn from data and improve their 
performance over time without being explicitly programmed. 
 
Types of Machine Learning 
 
1.  Supervised Learning Supervised learning uses labeled data to train 
    models. The model learns the relationship between input features and 
    target outputs. Examples include: 
 
-   Linear Regression 
-   Logistic Regressi